In [1]:
import pandas as pd

In [2]:
import arxiv


def set_client_search(query: str):
    client = arxiv.Client(
        page_size = 1000,   # 1リクエストでの抽出数。2000以下, default 100
        delay_seconds = 3.0,  # 遅延時間（秒）。rxivの規約上、1リクエスト以上/3秒はNG
    )

    search = arxiv.Search(
        query = query,  # 検索文字, str
        # id_list = [],   # 文献ID. List[str]
        max_results = 1000,  # 最大抽出数. 30000以下
        sort_by = arxiv.SortCriterion.Relevance,   # Relevance, LastUpdatedDate, SubmittedDate
        sort_order = arxiv.SortOrder.Descending,   # Ascending, Descending
    )
    return client, search

# 要約にRAGを含み、カテゴリ: AI、期間; 20210101-20250504で検索
query = ('("abs:"autonomous driving" )' 'AND (submittedDate:[202101010000 TO 202607252359])' )
client, search = set_client_search(query)
results = client.results(search)



In [4]:
import pandas as pd

def convert_to_str(vals):  # 文字列に変換
    if isinstance(vals, list):
        return list(map(str, vals))
    elif vals is None:
        return vals
    else:
        return str(vals)
        
def create_df(results):
    res0 = []
    for res in results:
        res_dict = dict(
            entry_id = res.entry_id,
            pdf_url = res.pdf_url,
            updated = res.updated, 
            published = res.published, 
            title = res.title, 
            authors = convert_to_str(res.authors),  # 文字列に変換
            summary = res.summary,
            comment = res.comment,
            journal_ref = res.journal_ref,
            doi = res.doi,
            primary_category = res.primary_category,
            categories = res.categories,
            links = convert_to_str(res.links),  # 文字列に変換
        )
        res0.append(res_dict)
    df = pd.DataFrame(res0)
    ndata = len(df)
    df['Relevance'] = [i/ndata for i in range(ndata, 0, -1)]  
    return df

df = create_df(results)



In [23]:
import numpy as np

def create_url(aid: str):
    apath = aid.split("/")[-1].split("v")[0]
    arxiv_id = f'ARXIV:{apath}'
    search_url = f'{semantic_base_url}/{arxiv_id}' + '?fields=' + fields 
    return arxiv_id

semantic_base_url = 'https://api.semanticscholar.org/graph/v1/paper'  # Semantic ScholarのベースURL
fields = 'citationCount,influentialCitationCount,referenceCount'  # サイトから取得する追加項
df['arxiv_id'] = df['entry_id'].apply(create_url)
tcol = fields.split(",") 
df[tcol] = np.nan  # 初期化

In [6]:
print(len(df), df.columns.tolist())

0 ['Relevance']


In [24]:
import requests
import time

HEADERS = {}   # APIキー取得後は {"x-api-key": API_KEY}

def post_batch(batch, max_retry=6):
    for attempt in range(max_retry):
        r = requests.post(semantic_base_url + '/batch',
                          params={'fields': fields}, json={"ids": batch},
                          headers=HEADERS, timeout=30)
        if r.status_code == 200:
            return r.json()
        if r.status_code in (429, 500, 502, 503, 504):
            wait = int(r.headers.get('Retry-After', 0)) or 5 * 2 ** attempt
            print(f'  {r.status_code}: retry in {wait}s (attempt {attempt+1})')
            time.sleep(wait)
            continue
        r.raise_for_status()
    return None


def get_citation_batch(df, start=0, batch_size=500):
    outs = []
    data = df['arxiv_id'].tolist()
    for i in range(start, len(data), batch_size):
        batch = data[i:i+batch_size]
        payload = post_batch(batch)
        if payload is None:
            print('Error: giving up. restart with start=', i)
            break
        rows = {pid: rec for pid, rec in zip(batch, payload) if rec is not None}
        if rows:
            outs.append(pd.DataFrame.from_dict(rows, orient="index"))
        time.sleep(5)
    if not outs:
        return df
    res = pd.concat(outs).rename_axis('arxiv_id').reset_index()
    return df.merge(res[['arxiv_id'] + tcol], on='arxiv_id', how='left')


start = 0  # 開始行
batch_size = 500  # バッチ数。500以下
df = get_citation_batch(df, start=start, batch_size=batch_size)



In [23]:
print(df.index.names)                      # ['arxiv_id'] になっているはず
print('arxiv_id' in df.columns)            # True のはず
print((df.index == df['arxiv_id']).all())  # 中身が同一かの確認

['arxiv_id']
True
True


In [25]:
df = df.reset_index(drop=True)
print(df.index.names, 'arxiv_id' in df.columns)   # [None] True になれば解消
df = get_citation_batch(df, start=0, batch_size=500)
print(len(df), df['citationCount'].notna().sum())

[None] True
Error: 429 restart with start= 0
1000 0


In [28]:
pd.set_option('display.max_columns', 50)

In [25]:
df = df.drop(columns=[c + '_x' for c in tcol])
df = df.rename(columns={c + '_y': c for c in tcol})

In [26]:
df.sort_values(['influentialCitationCount'],ascending=False)

,entry_id,pdf_url,updated,published,title,authors,summary,comment,journal_ref,doi,primary_category,categories,links,Relevance,arxiv_id,citationCount,influentialCitationCount,referenceCount
243,http://arxiv.org/abs/1711.03938v1,https://arxiv.org/pdf/1711.03938v1,2017-11-10 17:54:40+00:00,2017-11-10 17:54:40+00:00,CARLA: An Open Urban Driving Simulator,"[Alexey Dosovitskiy, German Ros, Felipe Codevi...","We introduce CARLA, an open-source simulator f...",Published at the 1st Conference on Robot Learn...,NaN,NaN,cs.LG,"[cs.LG, cs.AI, cs.CV, cs.RO]","[https://arxiv.org/abs/1711.03938v1, https://a...",0.757,ARXIV:1711.03938,6967.0,939.0,30.0
891,http://arxiv.org/abs/2203.17270v2,https://arxiv.org/pdf/2203.17270v2,2022-07-13 06:57:28+00:00,2022-03-31 17:59:01+00:00,BEVFormer: Learning Bird's-Eye-View Representa...,"[Zhiqi Li, Wenhai Wang, Hongyang Li, Enze Xie,...","3D visual perception tasks, including 3D detec...",Accepted to ECCV 2022,NaN,NaN,cs.CV,[cs.CV],"[https://arxiv.org/abs/2203.17270v2, https://a...",0.109,ARXIV:2203.17270,1993.0,386.0,57.0
656,http://arxiv.org/abs/1805.04687v2,https://arxiv.org/pdf/1805.04687v2,2020-04-08 09:25:06+00:00,2018-05-12 09:24:21+00:00,BDD100K: A Diverse Driving Dataset for Heterog...,"[Fisher Yu, Haofeng Chen, Xin Wang, Wenqi Xian...","Datasets drive vision progress, yet existing d...",Published at IEEE Conference on Computer Visio...,NaN,NaN,cs.CV,[cs.CV],"[https://arxiv.org/abs/1805.04687v2, https://a...",0.344,ARXIV:1805.04687,3047.0,331.0,47.0
599,http://arxiv.org/abs/2303.12077v3,https://arxiv.org/pdf/2303.12077v3,2023-08-24 08:15:35+00:00,2023-03-21 17:59:22+00:00,VAD: Vectorized Scene Representation for Effic...,"[Bo Jiang, Shaoyu Chen, Qing Xu, Bencheng Liao...",Autonomous driving requires a comprehensive un...,Accepted to ICCV 2023. Code&Demos: https://git...,NaN,NaN,cs.RO,"[cs.RO, cs.CV]","[https://arxiv.org/abs/2303.12077v3, https://a...",0.401,ARXIV:2303.12077,738.0,141.0,52.0
615,http://arxiv.org/abs/1908.09492v1,https://arxiv.org/pdf/1908.09492v1,2019-08-26 06:27:01+00:00,2019-08-26 06:27:01+00:00,Class-balanced Grouping and Sampling for Point...,"[Benjin Zhu, Zhengkai Jiang, Xiangxin Zhou, Ze...",This report presents our method which wins the...,technical report,NaN,NaN,cs.CV,[cs.CV],"[https://arxiv.org/abs/1908.09492v1, https://a...",0.385,ARXIV:1908.09492,587.0,76.0,33.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
999,http://arxiv.org/abs/2509.23922v1,https://arxiv.org/pdf/2509.23922v1,2025-09-28 14:55:14+00:00,2025-09-28 14:55:14+00:00,DriveE2E: Closed-Loop Benchmark for End-to-End...,"[Haibao Yu, Wenxian Yang, Ruiyang Hao, Chuanye...",Closed-loop evaluation is increasingly critica...,End-to-End Autonomous Driving Simulation and B...,NaN,NaN,cs.CV,"[cs.CV, cs.RO]","[https://arxiv.org/abs/2509.23922v1, https://a...",0.001,ARXIV:2509.23922,6.0,0.0,52.0
182,http://arxiv.org/abs/2405.08466v1,https://arxiv.org/pdf/2405.08466v1,2024-05-14 09:42:21+00:00,2024-05-14 09:42:21+00:00,Work-in-Progress: Crash Course: Can (Under Att...,"[Francesco Marchiori, Alessandro Brighente, Ma...",Autonomous driving is a research direction tha...,Accepted at ACSW 2024,NaN,NaN,cs.CR,[cs.CR],"[https://arxiv.org/abs/2405.08466v1, https://a...",0.818,ARXIV:2405.08466,NaN,NaN,NaN
359,http://arxiv.org/abs/2412.03051v1,https://arxiv.org/pdf/2412.03051v1,2024-12-04 06:11:09+00:00,2024-12-04 06:11:09+00:00,Less is More: A Stealthy and Efficient Adversa...,"[Junchao Fan, Xuyang Lei, Xiaolin Chang, Jelen...",Despite significant advancements in deep reinf...,NaN,NaN,NaN,cs.LG,"[cs.LG, cs.AI]","[https://arxiv.org/abs/2412.03051v1, https://a...",0.641,ARXIV:2412.03051,NaN,NaN,NaN
538,http://arxiv.org/abs/2212.12347v1,https://arxiv.org/pdf/2212.12347v1,2022-12-23 13:59:45+00:00,2022-12-23 13:59:45+00:00,Technical Report: Automating Vehicle SOA Threa...,"[Yuri Gil Dantas, Simon Barner, Pei Ke, Vivek ...",While the adoption of Service-Oriented Archite...,NaN,NaN,NaN,cs.LO,[cs.LO],"[http

In [76]:
df.shape

(1000, 18)

In [60]:
def set_client_search(query: str):
    client = arxiv.Client(
        page_size = 1000,   # 1リクエストでの抽出数。2000以下, default 100
        delay_seconds = 3.0,  # 遅延時間（秒）。rxivの規約上、1リクエスト以上/3秒はNG
    )

    query = ('abs:"autonomous driving"''AND (abs:"VLA" OR abs:"vision language model") ')

    client = arxiv.Client(
        page_size = 100,      # 修正: 1000→100。1リクエストあたりの負荷を下げる
        delay_seconds = 3.0,
        num_retries = 5,      # 修正: 503を踏んだ際のリトライ回数を増やす（default 3）
    )
    search = arxiv.Search(
        query = query,
        max_results = 1000,
        sort_by = arxiv.SortCriterion.SubmittedDate,   # 修正: relevance→投稿日
        sort_order = arxiv.SortOrder.Descending,
    )
    return client, search

# 要約にRAGを含み、カテゴリ: AI、期間; 20210101-20250504で検索
query = ('("abs:"autonomous driving" )' 'AND("abs:"VLA"OR abs:"Vision language model" )''AND (submittedDate:[202101010000 TO 202607252359])' )
client, search = set_client_search(query)
results = client.results(search)

In [61]:
def convert_to_str(vals):  # 文字列に変換
    if isinstance(vals, list):
        return list(map(str, vals))
    elif vals is None:
        return vals
    else:
        return str(vals)
        
def create_df(results):
    res0 = []
    for res in results:
        res_dict = dict(
            entry_id = res.entry_id,
            pdf_url = res.pdf_url,
            updated = res.updated, 
            published = res.published, 
            title = res.title, 
            authors = convert_to_str(res.authors),  # 文字列に変換
            summary = res.summary,
            comment = res.comment,
            journal_ref = res.journal_ref,
            doi = res.doi,
            primary_category = res.primary_category,
            categories = res.categories,
            links = convert_to_str(res.links),  # 文字列に変換
        )
        res0.append(res_dict)
    df = pd.DataFrame(res0)
    ndata = len(df)
    df['Relevance'] = [i/ndata for i in range(ndata, 0, -1)]  
    return df

df = create_df(results)


In [56]:
import urllib.parse, urllib.request
import xml.etree.ElementTree as ET

NS = {'opensearch': 'http://a9.com/-/spec/opensearch/1.1/'}

def arxiv_total(query: str) -> int:
    url = ('http://export.arxiv.org/api/query?search_query='
           + urllib.parse.quote(query) + '&max_results=1')
    with urllib.request.urlopen(url) as r:
        root = ET.fromstring(r.read())
    return int(root.find('opensearch:totalResults', NS).text)

print(arxiv_total(query))

1250750


In [59]:
import time

tests = [
    '(abs:"VLA" OR abs:"vision language model") AND submittedDate:[202101010000 TO 202607252359]',
    'abs:"autonomous driving" AND abs:"vision language model" AND submittedDate:[202101010000 TO 202607252359]',
]
for q in tests:
    print(arxiv_total(q), '|', q)
    time.sleep(3)

12275 | (abs:"VLA" OR abs:"vision language model") AND submittedDate:[202101010000 TO 202607252359]
336 | abs:"autonomous driving" AND abs:"vision language model" AND submittedDate:[202101010000 TO 202607252359]


In [58]:
tests = [
    '(abs:"VLA" OR abs:"vision language model") AND submittedDate:[202101010000 TO 202607252359]',
    'abs:"autonomous driving" AND abs:"vision language model" AND submittedDate:[202101010000 TO 202607252359]',
]

In [62]:
import numpy as np

def create_url(aid: str):
    apath = aid.split("/")[-1].split("v")[0]
    arxiv_id = f'ARXIV:{apath}'
    search_url = f'{semantic_base_url}/{arxiv_id}' + '?fields=' + fields 
    return arxiv_id

semantic_base_url = 'https://api.semanticscholar.org/graph/v1/paper'  # Semantic ScholarのベースURL
fields = 'citationCount,influentialCitationCount,referenceCount'  # サイトから取得する追加項
df['arxiv_id'] = df['entry_id'].apply(create_url)
tcol = fields.split(",") 
df[tcol] = np.nan  # 初期化

In [63]:
import requests
import time

HEADERS = {}   # APIキー取得後は {"x-api-key": API_KEY}

def post_batch(batch, max_retry=6):
    for attempt in range(max_retry):
        r = requests.post(semantic_base_url + '/batch',
                          params={'fields': fields}, json={"ids": batch},
                          headers=HEADERS, timeout=30)
        if r.status_code == 200:
            return r.json()
        if r.status_code in (429, 500, 502, 503, 504):
            wait = int(r.headers.get('Retry-After', 0)) or 5 * 2 ** attempt
            print(f'  {r.status_code}: retry in {wait}s (attempt {attempt+1})')
            time.sleep(wait)
            continue
        r.raise_for_status()
    return None


def get_citation_batch(df, start=0, batch_size=500):
    outs = []
    data = df['arxiv_id'].tolist()
    for i in range(start, len(data), batch_size):
        batch = data[i:i+batch_size]
        payload = post_batch(batch)
        if payload is None:
            print('Error: giving up. restart with start=', i)
            break
        rows = {pid: rec for pid, rec in zip(batch, payload) if rec is not None}
        if rows:
            outs.append(pd.DataFrame.from_dict(rows, orient="index"))
        time.sleep(5)
    if not outs:
        return df
    res = pd.concat(outs).rename_axis('arxiv_id').reset_index()
    return df.merge(res[['arxiv_id'] + tcol], on='arxiv_id', how='left')


start = 0  # 開始行
batch_size = 500  # バッチ数。500以下
df = get_citation_batch(df, start=start, batch_size=batch_size)

  429: retry in 5s (attempt 1)


In [64]:
df = df.drop(columns=[c + '_x' for c in tcol])
df = df.rename(columns={c + '_y': c for c in tcol})

In [65]:
df.sort_values(['influentialCitationCount'],ascending=False)

,entry_id,pdf_url,updated,published,title,authors,summary,comment,journal_ref,doi,primary_category,categories,links,Relevance,arxiv_id,citationCount,influentialCitationCount,referenceCount
419,http://arxiv.org/abs/2312.14150v3,https://arxiv.org/pdf/2312.14150v3,2025-01-16 10:57:44+00:00,2023-12-21 18:59:12+00:00,DriveLM: Driving with Graph Visual Question An...,"[Chonghao Sima, Katrin Renz, Kashyap Chitta, L...",We study how vision-language models (VLMs) tra...,Accepted to ECCV 2024 as Oral paper,NaN,NaN,cs.CV,[cs.CV],"[https://arxiv.org/abs/2312.14150v3, https://a...",0.023310,ARXIV:2312.14150,608.0,85.0,101.0
417,http://arxiv.org/abs/2402.12289v5,https://arxiv.org/pdf/2402.12289v5,2024-06-25 17:55:35+00:00,2024-02-19 17:04:04+00:00,DriveVLM: The Convergence of Autonomous Drivin...,"[Xiaoyu Tian, Junru Gu, Bailin Li, Yicheng Liu...",A primary hurdle of autonomous driving in urba...,Project Page: https://tsinghua-mars-lab.github...,NaN,NaN,cs.CV,[cs.CV],"[https://arxiv.org/abs/2402.12289v5, https://a...",0.027972,ARXIV:2402.12289,539.0,41.0,78.0
296,http://arxiv.org/abs/2506.13757v3,https://arxiv.org/pdf/2506.13757v3,2025-11-05 23:46:20+00:00,2025-06-16 17:58:50+00:00,AutoVLA: A Vision-Language-Action Model for En...,"[Zewei Zhou, Tianhui Cai, Seth Z. Zhao, Yun Zh...",Recent advancements in Vision-Language-Action ...,NeurIPS 2025; Website link:https://autovla.git...,NaN,NaN,cs.CV,[cs.CV],"[https://arxiv.org/abs/2506.13757v3, https://a...",0.310023,ARXIV:2506.13757,222.0,27.0,104.0
352,http://arxiv.org/abs/2503.09594v1,https://arxiv.org/pdf/2503.09594v1,2025-03-12 17:58:06+00:00,2025-03-12 17:58:06+00:00,SimLingo: Vision-Only Closed-Loop Autonomous D...,"[Katrin Renz, Long Chen, Elahe Arani, Oleg Sin...",Integrating large language models (LLMs) into ...,CVPR 2025. 1st Place @ CARLA Challenge 2024. C...,NaN,NaN,cs.CV,"[cs.CV, cs.RO]","[https://arxiv.org/abs/2503.09594v1, https://a...",0.179487,ARXIV:2503.09594,152.0,26.0,68.0
338,http://arxiv.org/abs/2504.04348v2,https://arxiv.org/pdf/2504.04348v2,2025-04-16 15:00:11+00:00,2025-04-06 03:54:21+00:00,OmniDrive: A Holistic Vision-Language Dataset ...,"[Shihao Wang, Zhiding Yu, Xiaohui Jiang, Shiyi...",The advances in vision-language models (VLMs) ...,Mistaken resubmission. The original version is...,NaN,NaN,cs.CV,[cs.CV],"[https://arxiv.org/abs/2504.04348v2, https://a...",0.212121,ARXIV:2504.04348,166.0,26.0,59.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139,http://arxiv.org/abs/2602.10458v1,https://arxiv.org/pdf/2602.10458v1,2026-02-11 02:56:04+00:00,2026-02-11 02:56:04+00:00,Found-RL: foundation model-enhanced reinforcem...,"[Yansong Qu, Zihao Sheng, Zilin Huang, Jiancon...",Reinforcement Learning (RL) has emerged as a d...,39 pages,NaN,NaN,cs.AI,"[cs.AI, cs.LG]","[https://arxiv.org/abs/2602.10458v1, https://a...",0.675991,ARXIV:2602.10458,3.0,0.0,21.0
138,http://arxiv.org/abs/2602.13329v1,https://arxiv.org/pdf/2602.13329v1,2026-02-11 07:08:33+00:00,2026-02-11 07:08:33+00:00,HiST-VLA: A Hierarchical Spatio-Temporal Visio...,"[Yiru Wang, Zichong Gu, Yu Gao, Anqing Jiang, ...",Vision-Language-Action (VLA) models offer prom...,NaN,NaN,NaN,cs.CV,"[cs.CV, cs.AI, cs.RO]","[https://arxiv.org/abs/2602.13329v1, https://a...",0.678322,ARXIV:2602.13329,3.0,0.0,41.0
213,http://arxiv.org/abs/2512.00021v2,https://arxiv.org/pdf/2512.00021v2,2026-03-23 16:53:27+00:00,2025-10-31 18:05:02+00:00,Foundation Models for Trajectory Planning in A...,"[Kemal Oksuz, Alexandru Buburuzan, Anthony Kni...",The emergence of multi-modal foundation models...,Accepted to TMLR (Survey Certification),NaN,NaN,cs.RO,"[cs.RO, cs.CV]","[https://arxiv.org/abs/2512.00021v2, https://a...",0.503497,ARXIV:2512.00021,1.0,0.0,183.0
228,http://arxiv.org/abs/2510.00060v3,https://arxiv.org/pdf/2510.00060v3,2026-02-27 09:00:30+00:00,2025-09-29 05:14:18+00:00,Less is More: Lean yet Powerful Vision-Languag...,"[Sheng Yang, Tong Zhan, Guancheng Chen, Yanfen...","In this work, we reconceptualize auto

In [21]:
client = arxiv.Client(
    page_size = 100,
    delay_seconds = 3.0,
    num_retries = 10,      # 間欠障害が前提なので回数を確保する
)
search = arxiv.Search(
    query = 'abs:"autonomous driving"',
    max_results = 1000,
    sort_by = arxiv.SortCriterion.Relevance,
    sort_order = arxiv.SortOrder.Descending,
)
df = create_df(client.results(search))
print(len(df))

1000


In [15]:
print(df['published'].min(), df['published'].max())
print(df[['citationCount', 'influentialCitationCount']].describe())
print(df['citationCount'].isna().sum(), '/', len(df))

2026-07-08 15:58:57+00:00 2026-07-24 17:56:54+00:00
       citationCount  influentialCitationCount
count     926.000000                926.000000
mean        0.078834                  0.004320
std         1.049708                  0.065617
min         0.000000                  0.000000
25%         0.000000                  0.000000
50%         0.000000                  0.000000
75%         0.000000                  0.000000
max        31.000000                  1.000000
74 / 1000


In [22]:
print(df['published'].dt.year.value_counts().sort_index())

published
2015      2
2016      2
2017     12
2018     31
2019     44
2020     50
2021     83
2022     72
2023    136
2024    224
2025    243
2026    101
Name: count, dtype: int64


In [32]:
NS = {'opensearch': 'http://a9.com/-/spec/opensearch/1.1/'}
for y in range(2015, 2027):
    q = f'abs:"autonomous driving" AND submittedDate:[{y}01010000 TO {y}12312359]'
    print(y, arxiv_total(q))
    time.sleep(3)

2015 11
2016 37
2017 90
2018 227
2019 362
2020 559
2021 725
2022 849
2023 1160
2024 1640
2025 2011
2026 1119


In [28]:
def arxiv_total(query: str) -> int:
    url = ('http://export.arxiv.org/api/query?search_query='
           + urllib.parse.quote(query) + '&max_results=1')
    with urllib.request.urlopen(url) as r:
        root = ET.fromstring(r.read())
    return int(root.find('opensearch:totalResults', NS).text)

In [30]:
import urllib.parse, urllib.request
import xml.etree.ElementTree as ET